In [ ]:
from functools import reduce
from hashlib import md5
import urllib.parse
import time
import requests

mixinKeyEncTab = [
    46, 47, 18, 2, 53, 8, 23, 32, 15, 50, 10, 31, 58, 3, 45, 35, 27, 43, 5, 49,
    33, 9, 42, 19, 29, 28, 14, 39, 12, 38, 41, 13, 37, 48, 7, 16, 24, 55, 40,
    61, 26, 17, 0, 1, 60, 51, 30, 4, 22, 25, 54, 21, 56, 59, 6, 63, 57, 62, 11,
    36, 20, 34, 44, 52
]

def getMixinKey(orig: str):
    '对 imgKey 和 subKey 进行字符顺序打乱编码'
    return reduce(lambda s, i: s + orig[i], mixinKeyEncTab, '')[:32]

def encWbi(params: dict, img_key: str, sub_key: str):
    '为请求参数进行 wbi 签名'
    mixin_key = getMixinKey(img_key + sub_key)
    curr_time = round(time.time())
    params['wts'] = curr_time                                   # 添加 wts 字段
    params = dict(sorted(params.items()))                       # 按照 key 重排参数
    # 过滤 value 中的 "!'()*" 字符
    params = {
        k : ''.join(filter(lambda chr: chr not in "!'()*", str(v)))
        for k, v
        in params.items()
    }
    query = urllib.parse.urlencode(params)                      # 序列化参数
    wbi_sign = md5((query + mixin_key).encode()).hexdigest()    # 计算 w_rid
    params['w_rid'] = wbi_sign
    return params

def getWbiKeys() -> tuple[str, str]:
    '获取最新的 img_key 和 sub_key'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
        'Referer': 'https://www.bilibili.com/'
    }
    resp = requests.get('https://api.bilibili.com/x/web-interface/nav', headers=headers)
    resp.raise_for_status()
    json_content = resp.json()
    img_url: str = json_content['data']['wbi_img']['img_url']
    sub_url: str = json_content['data']['wbi_img']['sub_url']
    img_key = img_url.rsplit('/', 1)[1].split('.')[0]
    sub_key = sub_url.rsplit('/', 1)[1].split('.')[0]
    return img_key, sub_key

img_key, sub_key = getWbiKeys()

signed_params = encWbi(
    params={
        'foo': '114',
        'bar': '514',
        'baz': 1919810
    },
    img_key=img_key,
    sub_key=sub_key
)
query = urllib.parse.urlencode(signed_params)
print(signed_params)
print(query)

In [1]:
url = 'https://api.bilibili.com/x/space/wbi/arc/search?pn=1&ps=40&tid=0&special_type=&order=pubdate&mid=3493140235290955&index=0&keyword=&order_avoided=true&platform=web&web_location=333.1387&dm_img_list=[%7B%22x%22:1699,%22y%22:-211,%22z%22:0,%22timestamp%22:548796,%22k%22:67,%22type%22:0%7D,%7B%22x%22:1437,%22y%22:-311,%22z%22:80,%22timestamp%22:548897,%22k%22:60,%22type%22:0%7D,%7B%22x%22:1312,%22y%22:-361,%22z%22:98,%22timestamp%22:548998,%22k%22:78,%22type%22:0%7D,%7B%22x%22:1335,%22y%22:-245,%22z%22:164,%22timestamp%22:549098,%22k%22:105,%22type%22:0%7D,%7B%22x%22:1244,%22y%22:-266,%22z%22:93,%22timestamp%22:549199,%22k%22:102,%22type%22:0%7D,%7B%22x%22:1280,%22y%22:-212,%22z%22:259,%22timestamp%22:549299,%22k%22:114,%22type%22:0%7D,%7B%22x%22:1363,%22y%22:-185,%22z%22:510,%22timestamp%22:549404,%22k%22:79,%22type%22:0%7D,%7B%22x%22:920,%22y%22:-628,%22z%22:90,%22timestamp%22:549505,%22k%22:119,%22type%22:0%7D,%7B%22x%22:861,%22y%22:-688,%22z%22:34,%22timestamp%22:549663,%22k%22:73,%22type%22:0%7D,%7B%22x%22:863,%22y%22:-683,%22z%22:73,%22timestamp%22:549763,%22k%22:116,%22type%22:0%7D,%7B%22x%22:1096,%22y%22:-444,%22z%22:311,%22timestamp%22:549865,%22k%22:78,%22type%22:0%7D,%7B%22x%22:1977,%22y%22:462,%22z%22:1140,%22timestamp%22:549966,%22k%22:104,%22type%22:0%7D,%7B%22x%22:1159,%22y%22:-327,%22z%22:235,%22timestamp%22:550068,%22k%22:79,%22type%22:0%7D,%7B%22x%22:1762,%22y%22:283,%22z%22:817,%22timestamp%22:550170,%22k%22:96,%22type%22:0%7D,%7B%22x%22:2019,%22y%22:542,%22z%22:1068,%22timestamp%22:550272,%22k%22:120,%22type%22:0%7D,%7B%22x%22:2535,%22y%22:1059,%22z%22:1581,%22timestamp%22:550480,%22k%22:105,%22type%22:0%7D,%7B%22x%22:2359,%22y%22:889,%22z%22:1387,%22timestamp%22:550581,%22k%22:110,%22type%22:0%7D,%7B%22x%22:2363,%22y%22:943,%22z%22:1264,%22timestamp%22:550682,%22k%22:124,%22type%22:0%7D,%7B%22x%22:1244,%22y%22:-175,%22z%22:142,%22timestamp%22:550783,%22k%22:63,%22type%22:0%7D,%7B%22x%22:1348,%22y%22:-138,%22z%22:286,%22timestamp%22:550884,%22k%22:70,%22type%22:0%7D,%7B%22x%22:1304,%22y%22:-271,%22z%22:256,%22timestamp%22:550985,%22k%22:66,%22type%22:0%7D,%7B%22x%22:2420,%22y%22:544,%22z%22:1033,%22timestamp%22:551090,%22k%22:72,%22type%22:0%7D,%7B%22x%22:3433,%22y%22:1392,%22z%22:1851,%22timestamp%22:551191,%22k%22:99,%22type%22:0%7D,%7B%22x%22:1953,%22y%22:-494,%22z%22:255,%22timestamp%22:551293,%22k%22:99,%22type%22:0%7D,%7B%22x%22:1967,%22y%22:-536,%22z%22:253,%22timestamp%22:551399,%22k%22:83,%22type%22:0%7D,%7B%22x%22:3608,%22y%22:1155,%22z%22:1928,%22timestamp%22:551499,%22k%22:113,%22type%22:0%7D,%7B%22x%22:3987,%22y%22:1701,%22z%22:2404,%22timestamp%22:551599,%22k%22:100,%22type%22:0%7D,%7B%22x%22:4549,%22y%22:2263,%22z%22:2966,%22timestamp%22:551843,%22k%22:83,%22type%22:1%7D,%7B%22x%22:1823,%22y%22:-462,%22z%22:237,%22timestamp%22:552199,%22k%22:99,%22type%22:0%7D,%7B%22x%22:3573,%22y%22:1220,%22z%22:1869,%22timestamp%22:552300,%22k%22:124,%22type%22:0%7D,%7B%22x%22:3656,%22y%22:1219,%22z%22:1859,%22timestamp%22:552402,%22k%22:78,%22type%22:0%7D,%7B%22x%22:3862,%22y%22:1370,%22z%22:2023,%22timestamp%22:552502,%22k%22:63,%22type%22:0%7D,%7B%22x%22:5619,%22y%22:2470,%22z%22:3129,%22timestamp%22:552605,%22k%22:112,%22type%22:0%7D,%7B%22x%22:4376,%22y%22:-618,%22z%22:291,%22timestamp%22:552705,%22k%22:63,%22type%22:0%7D,%7B%22x%22:6359,%22y%22:1024,%22z%22:1687,%22timestamp%22:552806,%22k%22:101,%22type%22:0%7D,%7B%22x%22:7362,%22y%22:1740,%22z%22:2355,%22timestamp%22:552907,%22k%22:113,%22type%22:0%7D,%7B%22x%22:6680,%22y%22:879,%22z%22:1612,%22timestamp%22:553008,%22k%22:111,%22type%22:0%7D,%7B%22x%22:5959,%22y%22:116,%22z%22:879,%22timestamp%22:553112,%22k%22:60,%22type%22:0%7D,%7B%22x%22:8558,%22y%22:2698,%22z%22:3483,%22timestamp%22:553216,%22k%22:92,%22type%22:0%7D,%7B%22x%22:7325,%22y%22:1420,%22z%22:2247,%22timestamp%22:553317,%22k%22:68,%22type%22:0%7D,%7B%22x%22:5876,%22y%22:-438,%22z%22:829,%22timestamp%22:553486,%22k%22:65,%22type%22:0%7D,%7B%22x%22:5548,%22y%22:-1350,%22z%22:781,%22timestamp%22:553589,%22k%22:80,%22type%22:0%7D,%7B%22x%22:7298,%22y%22:-737,%22z%22:3504,%22timestamp%22:553690,%22k%22:64,%22type%22:0%7D,%7B%22x%22:4241,%22y%22:-4092,%22z%22:674,%22timestamp%22:553791,%22k%22:69,%22type%22:0%7D,%7B%22x%22:5482,%22y%22:-2843,%22z%22:1914,%22timestamp%22:554022,%22k%22:66,%22type%22:0%7D,%7B%22x%22:8488,%22y%22:342,%22z%22:4751,%22timestamp%22:554124,%22k%22:99,%22type%22:0%7D,%7B%22x%22:6787,%22y%22:-1332,%22z%22:3038,%22timestamp%22:554226,%22k%22:96,%22type%22:0%7D,%7B%22x%22:8284,%22y%22:166,%22z%22:4532,%22timestamp%22:554327,%22k%22:107,%22type%22:0%7D,%7B%22x%22:6847,%22y%22:-1274,%22z%22:3127,%22timestamp%22:554427,%22k%22:76,%22type%22:0%7D,%7B%22x%22:6940,%22y%22:-1185,%22z%22:3232,%22timestamp%22:554535,%22k%22:114,%22type%22:0%7D]&dm_img_str=V2ViR0wgMS4wIChPcGVuR0wgRVMgMi4wIENocm9taXVtKQ&dm_cover_img_str=QU5HTEUgKE5WSURJQSwgTlZJRElBIEdlRm9yY2UgR1RYIDEwNTAgKDB4MDAwMDFDODEpIERpcmVjdDNEMTEgdnNfNV8wIHBzXzVfMCwgRDNEMTEpR29vZ2xlIEluYy4gKE5WSURJQS&dm_img_inter=%7B%22ds%22:[%7B%22t%22:7,%22c%22:%22dnVpX2J1dHRvbiB2dWlfYnV0dG9uLS1hY3RpdmUgdnVpX2J1dHRvbi0tYWN0aXZlLWJsdWUgdnVpX2J1dHRvbi0tbm8tdHJhbnNpdGlvbiB2dWlfcGFnZW5hdGlvbi0tYnRuIHZ1aV9wYWdlbmF0aW9uLS1idG4tbn%22,%22p%22:[5970,4,10190],%22s%22:[482,652,964]%7D],%22wh%22:[5418,3876,70],%22of%22:[4504,6168,244]%7D&w_webid=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJzcG1faWQiOiIzMzMuMTM4NyIsImJ1dmlkIjoiNzNGNkVGNkMtMjA5RS01OTlBLTU5NTgtMzIxOTAwNjQwNUZFNDU1MTdpbmZvYyIsInVzZXJfYWdlbnQiOiJNb3ppbGxhLzUuMCAoV2luZG93cyBOVCAxMC4wOyBXaW42NDsgeDY0KSBBcHBsZVdlYktpdC81MzcuMzYgKEtIVE1MLCBsaWtlIEdlY2tvKSBDaHJvbWUvMTM2LjAuMC4wIFNhZmFyaS81MzcuMzYiLCJidXZpZF9mcCI6Ijk0MmQyZDRkOTI2YWJmMGI5YTA2YjY2YTZjYTc5OTMxIiwiYmlsaV90aWNrZXQiOiJleUpoYkdjaU9pSklVekkxTmlJc0ltdHBaQ0k2SW5Nd015SXNJblI1Y0NJNklrcFhWQ0o5LmV5SmxlSEFpT2pFM05EZzVNakk0TkRFc0ltbGhkQ0k2TVRjME9EWTJNelU0TVN3aWNHeDBJam90TVgwLnpEbkx5OTBFa1NHZEFDNm1fdjREZHRRYkJWaVRhWVFMb2FsOWtpZXlpeHMiLCJjcmVhdGVkX2F0IjoxNzQ4NzA0NDQwLCJ0dGwiOjg2NDAwLCJ1cmwiOiIvMzQ5MzE0MDIzNTI5MDk1NSIsInJlc3VsdCI6MCwiaXNzIjoiZ2FpYSIsImlhdCI6MTc0ODcwNDQ0MH0.g18_pWsJ1_imnCUnvPV2J3t91gHZYNhGK_BgjZZchQxAZNZ7RBzOgBneIjHC0Ct5UiANX7Lga_ZXbo6g8YPMxcz2eUNmvDEBZ7WlS_P4ubu2jLsX6GaSVE4kXnviJXgyY135cU_Aq1ggCz1EbAoLf3vksiBRi6X5yLzBhnlUFUsulsgIKXsB9yEHZ1IFWhH5QBoVe3nrO-cK4RRJ94IhsD8PKLI1CpbG11G5B0UsSYqaVq1GwYmhax-K07Y6zmCbRrRm_2uWJR0tFksG6TtNn7JLe6P4lJKjAKS8mbWUC9tqlYzZLSC4G8qqBBIsGLBlEkMNvEpeluKG2gKwvFVLCw&w_rid=b0e5dbd32d6094ae0c677fe16df64b7e&wts=1748704999'

In [2]:
url_arr = url.split("?")
base_url = url_arr[0]
params_concat_str = url_arr[1]

In [3]:
base_url

'https://api.bilibili.com/x/space/wbi/arc/search'

In [4]:
params_concat_str

'pn=1&ps=40&tid=0&special_type=&order=pubdate&mid=3493140235290955&index=0&keyword=&order_avoided=true&platform=web&web_location=333.1387&dm_img_list=[%7B%22x%22:1699,%22y%22:-211,%22z%22:0,%22timestamp%22:548796,%22k%22:67,%22type%22:0%7D,%7B%22x%22:1437,%22y%22:-311,%22z%22:80,%22timestamp%22:548897,%22k%22:60,%22type%22:0%7D,%7B%22x%22:1312,%22y%22:-361,%22z%22:98,%22timestamp%22:548998,%22k%22:78,%22type%22:0%7D,%7B%22x%22:1335,%22y%22:-245,%22z%22:164,%22timestamp%22:549098,%22k%22:105,%22type%22:0%7D,%7B%22x%22:1244,%22y%22:-266,%22z%22:93,%22timestamp%22:549199,%22k%22:102,%22type%22:0%7D,%7B%22x%22:1280,%22y%22:-212,%22z%22:259,%22timestamp%22:549299,%22k%22:114,%22type%22:0%7D,%7B%22x%22:1363,%22y%22:-185,%22z%22:510,%22timestamp%22:549404,%22k%22:79,%22type%22:0%7D,%7B%22x%22:920,%22y%22:-628,%22z%22:90,%22timestamp%22:549505,%22k%22:119,%22type%22:0%7D,%7B%22x%22:861,%22y%22:-688,%22z%22:34,%22timestamp%22:549663,%22k%22:73,%22type%22:0%7D,%7B%22x%22:863,%22y%22:-683,%22z%22

In [5]:

param_pair_arr = params_concat_str.split("&")
param_pair_arr

['pn=1',
 'ps=40',
 'tid=0',
 'special_type=',
 'order=pubdate',
 'mid=3493140235290955',
 'index=0',
 'keyword=',
 'order_avoided=true',
 'platform=web',
 'web_location=333.1387',
 'dm_img_list=[%7B%22x%22:1699,%22y%22:-211,%22z%22:0,%22timestamp%22:548796,%22k%22:67,%22type%22:0%7D,%7B%22x%22:1437,%22y%22:-311,%22z%22:80,%22timestamp%22:548897,%22k%22:60,%22type%22:0%7D,%7B%22x%22:1312,%22y%22:-361,%22z%22:98,%22timestamp%22:548998,%22k%22:78,%22type%22:0%7D,%7B%22x%22:1335,%22y%22:-245,%22z%22:164,%22timestamp%22:549098,%22k%22:105,%22type%22:0%7D,%7B%22x%22:1244,%22y%22:-266,%22z%22:93,%22timestamp%22:549199,%22k%22:102,%22type%22:0%7D,%7B%22x%22:1280,%22y%22:-212,%22z%22:259,%22timestamp%22:549299,%22k%22:114,%22type%22:0%7D,%7B%22x%22:1363,%22y%22:-185,%22z%22:510,%22timestamp%22:549404,%22k%22:79,%22type%22:0%7D,%7B%22x%22:920,%22y%22:-628,%22z%22:90,%22timestamp%22:549505,%22k%22:119,%22type%22:0%7D,%7B%22x%22:861,%22y%22:-688,%22z%22:34,%22timestamp%22:549663,%22k%22:73,%22typ

In [6]:
params = {}
for param_pair in param_pair_arr:
    key_value = param_pair.split("=")
    key = key_value[0]
    value = key_value[1]
    params[key] = value

In [7]:
params

{'pn': '1',
 'ps': '40',
 'tid': '0',
 'special_type': '',
 'order': 'pubdate',
 'mid': '3493140235290955',
 'index': '0',
 'keyword': '',
 'order_avoided': 'true',
 'platform': 'web',
 'web_location': '333.1387',
 'dm_img_list': '[%7B%22x%22:1699,%22y%22:-211,%22z%22:0,%22timestamp%22:548796,%22k%22:67,%22type%22:0%7D,%7B%22x%22:1437,%22y%22:-311,%22z%22:80,%22timestamp%22:548897,%22k%22:60,%22type%22:0%7D,%7B%22x%22:1312,%22y%22:-361,%22z%22:98,%22timestamp%22:548998,%22k%22:78,%22type%22:0%7D,%7B%22x%22:1335,%22y%22:-245,%22z%22:164,%22timestamp%22:549098,%22k%22:105,%22type%22:0%7D,%7B%22x%22:1244,%22y%22:-266,%22z%22:93,%22timestamp%22:549199,%22k%22:102,%22type%22:0%7D,%7B%22x%22:1280,%22y%22:-212,%22z%22:259,%22timestamp%22:549299,%22k%22:114,%22type%22:0%7D,%7B%22x%22:1363,%22y%22:-185,%22z%22:510,%22timestamp%22:549404,%22k%22:79,%22type%22:0%7D,%7B%22x%22:920,%22y%22:-628,%22z%22:90,%22timestamp%22:549505,%22k%22:119,%22type%22:0%7D,%7B%22x%22:861,%22y%22:-688,%22z%22:34,%22t

In [8]:
from request_helper import request_get

In [9]:
res = request_get(base_url, params)
res

<Response [200]>

In [10]:
res.text


'{"code":-403,"message":"访问权限不足","ttl":1}'